# Transmission Network NPV Model - Interactive Analysis

This notebook provides interactive analysis and visualization for the
regulated high-voltage transmission network NPV model.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from transmission_npv.config import ModelConfig
from transmission_npv.model import TransmissionNPVModel
from transmission_npv.sensitivity import run_tornado_analysis, run_scenario_comparison
from transmission_npv.reports import format_summary

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Run Base Case Model

In [ ]:
config = ModelConfig()
model = TransmissionNPVModel(config)
model.run()

print(format_summary(model.summary()))

## 2. RAB Roll-Forward

In [ ]:
fig, ax = plt.subplots()
rab = model.rab_schedule
ax.fill_between(rab.index, rab['closing_rab'], alpha=0.3, label='Closing RAB')
ax.plot(rab.index, rab['closing_rab'], linewidth=2, label='Closing RAB')
ax.bar(rab.index, rab['capex_additions'], alpha=0.5, color='green', label='CAPEX Additions', width=0.8)
ax.bar(rab.index, -rab['regulatory_depreciation'], alpha=0.5, color='red', label='Depreciation', width=0.8)
ax.set_xlabel('Year')
ax.set_ylabel('$M')
ax.set_title('Regulated Asset Base Roll-Forward')
ax.legend()
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'${x:,.0f}M'))
plt.tight_layout()
plt.show()

## 3. Revenue & Cost Breakdown

In [ ]:
df = model.to_dataframe()
operating = df[df.index >= config.project.cod_year]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Revenue components
rev = model.revenue[model.revenue.index >= config.project.cod_year]
ax1.stackplot(rev.index, 
              rev['return_on_rab'], 
              rev['depreciation_allowance'],
              rev['opex_allowance'],
              labels=['Return on RAB', 'Depreciation', 'OPEX Allowance'],
              alpha=0.7)
ax1.set_title('Allowed Revenue Components')
ax1.set_xlabel('Year')
ax1.set_ylabel('$M')
ax1.legend(loc='upper left')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'${x:,.0f}M'))

# OPEX components
opex = model.opex_projection[model.opex_projection.index >= config.project.cod_year]
ax2.stackplot(opex.index,
              opex['maintenance'],
              opex['staffing'],
              opex['grid_losses'],
              opex['insurance'],
              opex['regulatory_compliance'],
              labels=['Maintenance', 'Staffing', 'Grid Losses', 'Insurance', 'Reg. Compliance'],
              alpha=0.7)
ax2.set_title('OPEX Breakdown')
ax2.set_xlabel('Year')
ax2.set_ylabel('$M')
ax2.legend(loc='upper left')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'${x:,.0f}M'))

plt.tight_layout()
plt.show()

## 4. Cash Flow Waterfall

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

# FCFF
colors = ['green' if v >= 0 else 'red' for v in df['fcff']]
ax1.bar(df.index, df['fcff'], color=colors, alpha=0.7)
ax1.set_title('Free Cash Flow to Firm (FCFF)')
ax1.set_xlabel('Year')
ax1.set_ylabel('$M')
ax1.axhline(y=0, color='black', linewidth=0.5)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'${x:,.0f}M'))

# Cumulative FCFF
cumulative = df['fcff'].cumsum()
ax2.plot(df.index, cumulative, linewidth=2, color='navy')
ax2.fill_between(df.index, cumulative, alpha=0.2, color='navy')
ax2.axhline(y=0, color='red', linewidth=1, linestyle='--')
ax2.set_title('Cumulative FCFF (Payback Visualization)')
ax2.set_xlabel('Year')
ax2.set_ylabel('$M')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'${x:,.0f}M'))

plt.tight_layout()
plt.show()

## 5. Sensitivity Analysis - Tornado Chart

In [ ]:
tornado = run_tornado_analysis(
    config,
    lambda cfg: TransmissionNPVModel(cfg).run().valuation,
    parameters=[
        'capex.cost_per_km',
        'capex.cost_per_substation',
        'funding.cost_of_equity',
        'funding.cost_of_debt',
        'regulatory.allowed_return_on_equity',
        'opex.staffing_base',
    ],
    variation_pct=0.20,
)

fig, ax = plt.subplots(figsize=(12, 6))
base_npv = tornado['base_npv'].iloc[0]

y_pos = range(len(tornado))
low_bars = tornado['low_npv'] - base_npv
high_bars = tornado['high_npv'] - base_npv

ax.barh(y_pos, high_bars, align='center', color='steelblue', alpha=0.7, label='High (+20%)')
ax.barh(y_pos, low_bars, align='center', color='coral', alpha=0.7, label='Low (-20%)')
ax.set_yticks(y_pos)
ax.set_yticklabels(tornado['parameter'])
ax.axvline(x=0, color='black', linewidth=0.5)
ax.set_xlabel('Change in NPV ($M)')
ax.set_title(f'Tornado Chart - NPV Sensitivity (Base NPV: ${base_npv:,.0f}M)')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Scenario Comparison

In [ ]:
scenarios = {
    'Base Case': ModelConfig.from_json('../scenarios/base_case.json'),
    'Upside': ModelConfig.from_json('../scenarios/upside.json'),
    'Downside': ModelConfig.from_json('../scenarios/downside.json'),
}

# Run all scenarios
results = {}
for name, cfg in scenarios.items():
    m = TransmissionNPVModel(cfg).run()
    results[name] = m.summary()

comparison = pd.DataFrame(results).T
display_cols = ['npv_project_$M', 'npv_equity_$M', 'irr_project', 
                'total_initial_capex_$M', 'wacc_post_tax', 'simple_payback_years']
comparison[display_cols]

In [ ]:
# Compare scenario cash flows
fig, ax = plt.subplots(figsize=(14, 6))

for name, cfg in scenarios.items():
    m = TransmissionNPVModel(cfg).run()
    cf = m.to_dataframe()
    ax.plot(cf.index, cf['fcff'].cumsum(), linewidth=2, label=name)

ax.axhline(y=0, color='red', linewidth=1, linestyle='--')
ax.set_title('Cumulative FCFF by Scenario')
ax.set_xlabel('Year')
ax.set_ylabel('$M')
ax.legend()
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'${x:,.0f}M'))
plt.tight_layout()
plt.show()

## 7. Debt Profile

In [ ]:
debt = model.debt_schedule

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.fill_between(debt.index, debt['closing_debt'], alpha=0.3, color='navy')
ax1.plot(debt.index, debt['closing_debt'], linewidth=2, color='navy')
ax1.set_title('Debt Balance Over Time')
ax1.set_xlabel('Year')
ax1.set_ylabel('$M')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'${x:,.0f}M'))

ax2.bar(debt.index, debt['interest_expense'], alpha=0.7, color='coral')
ax2.set_title('Annual Interest Expense')
ax2.set_xlabel('Year')
ax2.set_ylabel('$M')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'${x:,.0f}M'))

plt.tight_layout()
plt.show()